In [ ]:
#This script is used to sweep the threshold for the trained model. It is a modified version of test_bash.py, which is used to test the trained model. The main difference is that this script will sweep the threshold from 0.1 to 0.9 with a step of 0.1, and save the results in a specified directory.
#Their original code cannot be normally run. 
#issue1: The original code uses a hard-coded threshold of 0.5. Here we made it a variable that can be set by the user.
#issue2: The way of reading test files is also modified to be more flexible.  
#issue3: Saving results often fail using the original code. Here we made it more robust.
#Allows to do multiple threshold testing for different models by running the model once. 


import importlib
import json
import os
import sys
from os.path import abspath, dirname

import numpy as np

In [ ]:
DeepETPickerHome = "path to base deepETpicker/bin"
DeepETPickerHome = os.path.split(DeepETPickerHome)[0]
sys.path.append(DeepETPickerHome)
sys.path.append(os.path.split(DeepETPickerHome)[0])
train = importlib.import_module(".train", package=os.path.split(DeepETPickerHome)[1])
option = importlib.import_module(f".options.option", package=os.path.split(DeepETPickerHome)[1])


/data/biosoftware/miniconda3/miniconda3/envs/deepET/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# test = importlib.import_module(".test", package=os.path.split(DeepETPickerHome)[1])
# option = importlib.import_module(
#     f".options.option", package=os.path.split(DeepETPickerHome)[1]
# )
import ast
import shlex

# fake command line arguments to continue using the same code as in train.py
argv_line = "<python_path> <test_bash.py> --train_configs <traiin_config_path> --checkpoints <checkpoint_path>"
# Some logs use argvg_line; fall back to argv_line when needed.
raw_argv_line = globals().get("argvg_line", argv_line)

def parse_argv_line(raw):
    raw = str(raw).strip()
    if raw.startswith("[") and raw.endswith("]"):
        # Supports repr-style inputs like [PosixPath('...'), '--flag', 'value']
        parsed = eval(
            raw,
            {"__builtins__": {}},
            {
                "PosixPath": lambda p: p,
                "WindowsPath": lambda p: p,
                "Path": lambda p: p,
            },
        )
        return [str(item) for item in parsed]
    return shlex.split(raw)

argv = parse_argv_line(raw_argv_line)
if argv and os.path.basename(argv[0]).startswith("python"):
    sys.argv = argv[1:]
else:
    sys.argv = argv

options = option.BaseOptions()
args = options.gather_options()

In [ ]:
with open("the path to the training config", "r") as f:
    cfg = json.loads("".join(f.readlines()).lstrip("train_configs="))

In [7]:
args.use_bg = True
args.use_IP = True
args.use_coord = True
args.test_use_pad = True
args.use_seg = True
args.meanPool_NMS = True
args.f_maps = [24, 48, 72, 108]
args.num_classes = cfg["num_cls"]
train_cls_num = cfg["num_cls"]
if args.num_classes == 1:
    args.use_sigmoid = True
    args.use_softmax = False
else:
    train_cls_num = train_cls_num + 1
    args.use_sigmoid = False
    args.use_softmax = True
args.batch_size = cfg["batch_size"]
args.block_size = cfg["patch_size"]
args.val_batch_size = args.batch_size
args.val_block_size = args.block_size
args.pad_size = [cfg["padding_size"]]
args.learning_rate = cfg["lr"]
args.max_epoch = cfg["max_epochs"]
args.threshold = cfg["seg_thresh"]
args.gpu_id = [int(i) for i in cfg["gpu_ids"].split(",")]
args.test_mode = "test_only"
args.out_name = "hsp_res"
args.de_duplication = True
args.de_dup_fmt = "fmt4"
args.mini_dist = sorted([int(i) // 2 + 1 for i in cfg["ocp_diameter"].split(",")])[
    0
]
# args.data_split = [0, 1, 0, 1, 0, 1]
args.configs = cfg
args.num_classes = train_cls_num

# test_idxs
dset_list = np.array(
    [
        i[: -(len(i.split(".")[-1]) + 1)]
        for i in os.listdir(cfg["tomo_path"])
        if cfg["tomo_format"] in i
    ]
)
dset_num = dset_list.shape[0]
num_name = np.concatenate(
    [np.arange(dset_num).reshape(-1, 1), dset_list.reshape(-1, 1)], axis=1
)
np.savetxt(
    os.path.join(cfg["tomo_path"], "num_name.csv"),
    num_name,
    delimiter="\t",
    fmt="%s",
    newline="\n",
)

# tomo_list = [i for i in os.listdir(cfg[f"{cfg['base_path']}/data_std"]) if cfg['tomo_format'] in i]
tomo_list = np.loadtxt(
    f"{cfg['base_path']}/data_std/num_name.csv", delimiter="\t", dtype=str
)
args.test_idxs = np.arange(len(tomo_list))

for k, v in sorted(vars(args).items()):
    print(k, "=", v)

# Testing


CL_DA_fmt = mean
FL_a0 = 0.1
FL_a1 = 0.9
Gau_nms = False
Gau_num = 2
HRNet_c = 12
JudgeInDilation = False
NoBG = False
ResearchTitle = None
Sel_Referance = False
T = 0.5
act = relu
add_dropout_layer = False
alpha = 0.7
batch_size = 8
beta = 0.3
block_size = 72
border_value = 0
cent_feats = None
cent_kernel = 1
cfg_save_path = .
checkpoints = /data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/runs/ribosome_fold1_train_fold2_test/ribosome_fold1_train_fold2_test_ResUNet_BlockSize72_DiceLoss_MaxEpoch24_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/checkpoints/epoch=16-step=10370.ckpt
checkpoints_version = None
ck_mode = 
class_checkpoints = None
configs = {'dset_name': 'hsp60_train_holdout_5', 'base_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60', 'coord_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/coords', 'coord_format': '.coords', 'tomo_path': 

In [8]:
import ast
import importlib
import json
import os
import sys
import time
from glob import glob

import matplotlib.pyplot as plt
import mrcfile
import numpy as np

# from dataset.dataloader import Dataset_ClsBased
import pandas as pd
import pytorch_lightning as pl
import torch
from dataset.dataloader_DynamicLoad import Dataset_ClsBased
from model.model_loader import get_model
from options.option import BaseOptions
from pytorch_lightning import Trainer
from torch import nn
from torch.utils.data import DataLoader
from utils.misc import cal_metrics_NMS_OneCls, combine, de_dup, get_centroids

In [ ]:

for dir_name in [list, of, ids, to, be, tested]:
    args.checkpoints = path_to_the_checkpoint_file
    model_name = (
        args.checkpoints.split("/")[-4]
        + "_"
        + args.checkpoints.split("/")[-1].split("-")[0]
    )
   
    cfg = args.configs
    start_time = time.time()
    # args.data_split[-2] = test_idx
    # args.data_split[-1] = test_idx + 1
    sp = [0, 0, 0, 0, dir_name, dir_name+1]
    args.data_split = sp
    
    num_name = pd.read_csv(
        os.path.join(cfg["tomo_path"], "num_name.csv"), sep="\t", header=None, index_col=0
    )
    # print(num_name)
    # dir_list = num_name.iloc[:, 1]
    # dir_name = dir_list[args.data_split[-2]]
    dir_name = num_name.loc[dir_name, 1]
    print(dir_name)
    # continue
    tomo_file = glob(cfg["tomo_path"] + "/*%s" % cfg["tomo_format"])[0]
    # tomo_file = "/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/data_std/" + dir_name+".mrc"
    data_file = mrcfile.open(tomo_file, permissive=True)
    data_shape = data_file.data.shape
    print(data_shape)
    dataset = cfg["dset_name"]

    if args.use_seg:
        for pad_size in args.pad_size:

            class UNetTest(pl.LightningModule):
                def __init__(self):
                    super(UNetTest, self).__init__()
                    self.thres = args.threshold
                    self.p1 = 0
                    self.model = get_model(args)
                    self.partical_volume = (
                        4 / 3 * np.pi * (cfg["label_diameter"] / 2) ** 3
                    )
                    self.num_classes = args.num_classes
                    self._epoch_outputs = []

                def forward(self, x):
                    return self.model(x)

                def test_step(self, test_batch, batch_idx):
                    # print("test")
                    with torch.no_grad():
                        img, label, index = test_batch
                        index = torch.cat(
                            [i.view(1, -1) for i in index], dim=0
                        ).permute(1, 0)
                        if args.use_paf:
                            seg_output, paf_output, logsigma1 = self.forward(img)
                        else:
                            seg_output = self.forward(img)

                        if args.test_use_pad:
                            # print("test")
                            mp_num = int(
                                sorted(
                                    [int(i) for i in cfg["ocp_diameter"].split(",")]
                                )[-1]
                                / (args.meanPool_kernel - 1)
                                + 1
                            )
                            for thres in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85,0.9, 0.95]:
                                if args.num_classes > 1:
                                    t = self._nms_v2(
                                        seg_output[:, 1:],
                                        kernel=args.meanPool_kernel,
                                        mp_num=mp_num,
                                        positions=index,
                                        thres = thres
                                    )
                                    # return 
                                else:
                                    t = self._nms_v2(
                                        seg_output[:, :],
                                        kernel=args.meanPool_kernel,
                                        mp_num=mp_num,
                                        positions=index,
                                        thres = thres
                                    )
                                    # return 
                                if t is not None:
                                    self._epoch_outputs[thres].append(t)

                def test_step_end(self, outputs):
                    if outputs is not None:
                        # print("input")
                        self._epoch_outputs.append(outputs)
                    return outputs

                def on_test_epoch_start(self):
                    
                    self._epoch_outputs = {} 
                    for i in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85,0.9, 0.95]:
                        self._epoch_outputs[i] = []

                def on_test_epoch_end(self):
                    print("here at end")
                    with torch.no_grad():
                        if True:
                            # print(self._epoch_outputs)
                            for thres in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
                                print("dealing, thres", thres)
                                if len(self._epoch_outputs[thres]) == 0:
                                    continue
                                coords_out = (
                                    torch.cat(self._epoch_outputs[thres], dim=0)
                                .detach()
                                .cpu()
                                .numpy()
                            )
                                print("coords_out:", coords_out.shape)
                                if args.de_duplication:
                                    centroids = de_dup(coords_out, args)
                                out_dir = (
                                    "/".join(args.checkpoints.split("/")[:-2])
                                    + f"/{args.out_name}"
                                )
                                # os.makedirs(
                                #     os.path.join(out_dir, "Coords_withArea"),
                                #     exist_ok=True,
                                # )
                
                                sweep_res_path = "path for saving the results"
                                np.savetxt(
                                    sweep_res_path
                                    + "/"
                                    + str(thres)
                                    + "_"
                                    + str(dir_name)
                                    + "_coords.txt",
                                    centroids.astype(float),
                                    fmt="%s",
                                    delimiter="\t",
                                )
                                # np.savetxt(
                                #     os.path.join(
                                #         out_dir, "Coords_withArea", str(dir_name) + ".coords"
                                #     ),
                                #     centroids.astype(float),
                                #     fmt="%s",
                                #     delimiter="\t",
                                # )

                                coords = centroids[:, 0:4]
                                # os.makedirs(
                                #     os.path.join(out_dir, "Coords_All"), exist_ok=True
                                # )
                                np.savetxt(
                                    sweep_res_path
                                    + "/"
                                    + str(thres)
                                    + "_"
                                    + str(dir_name)
                                    + "_coords_noarea.txt",
                                    centroids.astype(float),
                                    fmt="%s",
                                    delimiter="\t",
                                )
                                # np.savetxt(
                                #     os.path.join(
                                #         out_dir, "Coords_All", dir_name + ".coords"
                                #     ),
                                #     coords.astype(int),
                                #     fmt="%s",
                                #     delimiter="\t",
                                # )

                def test_dataloader(self):
                    if args.test_mode == "test":
                        test_dataset = Dataset_ClsBased(
                            mode="test",
                            block_size=args.block_size,
                            num_class=args.num_classes,
                            random_num=args.random_num,
                            use_bg=args.use_bg,
                            test_use_pad=args.test_use_pad,
                            pad_size=pad_size,
                            data_split=sp,
                            cfg=cfg,
                            args=args,
                        )
                        test_dataloader = DataLoader(
                            test_dataset,
                            shuffle=False,
                            batch_size=args.batch_size,
                            num_workers=8 if args.batch_size >= 32 else 4,
                            pin_memory=False,
                        )

                        self.len_block = test_dataset.test_len
                        self.data_shape = test_dataset.data_shape
                        self.occupancy_map = test_dataset.occupancy_map
                        self.gt_coords = test_dataset.gt_coords
                        self.dir_name = test_dataset.dir_name
                        return test_dataloader
                    elif args.test_mode == "test_only":
                        test_dataset = Dataset_ClsBased(
                            mode="test_only",
                            block_size=args.block_size,
                            num_class=args.num_classes,
                            random_num=args.random_num,
                            use_bg=args.use_bg,
                            data_split=args.data_split,
                            test_use_pad=args.test_use_pad,
                            pad_size=pad_size,
                            cfg=cfg,
                            args=args,
                        )
                        if args.batch_size <= 32:
                            num_work = 4
                        elif args.batch_size <= 64:
                            num_work = 8
                        elif args.batch_size <= 128:
                            num_work = 8
                        else:
                            num_work = 16
                        test_dataloader = DataLoader(
                            test_dataset,
                            shuffle=False,
                            batch_size=args.batch_size,
                            num_workers=num_work,
                            pin_memory=False,
                        )
                        self.len_block = test_dataset.test_len
                        self.data_shape = test_dataset.data_shape
                        self.dir_name = test_dataset.dir_name
                        return test_dataloader

                def _nms_v2(self, pred, kernel=3, mp_num=5, positions=None, thres = 0.5):
                    # print("test")
                    # if self.thres is None:
                    pred = torch.where(pred > thres, 1, 0)
                    # else:
                    #     if self.p1 == 0:
                    #         print("use_thres")
                    #         self.p1 = 1
                    #     pred = torch.where(pred > self.thres, 1, 0)
                    # print(pred)
                    meanPool = nn.AvgPool3d(kernel, 1, kernel // 2).cuda()
                    maxPool = nn.MaxPool3d(kernel, 1, kernel // 2).cuda()
                    hmax = pred.clone().float()
                    for _ in range(mp_num):
                        hmax = meanPool(hmax)
                    pred = hmax.clone()
                    hmax = maxPool(hmax)
                    keep = ((hmax == pred).float()) * ((pred > 0.1).float())
                    coords = keep.nonzero()  # [N, 5]
                    coords = coords[coords[:, 2] >= args.pad_size[0]]
                    coords = coords[
                        coords[:, 2] <= args.block_size - args.pad_size[0]
                    ]
                    coords = coords[coords[:, 3] >= args.pad_size[0]]
                    coords = coords[
                        coords[:, 3] <= args.block_size - args.pad_size[0]
                    ]
                    coords = coords[coords[:, 4] >= args.pad_size[0]]
                    coords = coords[
                        coords[:, 4] <= args.block_size - args.pad_size[0]
                    ]

                    try:
                        h_val = torch.cat(
                            [
                                hmax[
                                    item[0],
                                    item[1],
                                    item[2],
                                    item[3] : item[3] + 1,
                                    item[4] : item[4] + 1,
                                ]
                                for item in coords
                            ],
                            dim=0,
                        )
                        leftTop_coords = (
                            positions[coords[:, 0]]
                            - (args.block_size // 2)
                            - args.pad_size[0]
                        )
                        coords[:, 2:5] = coords[:, 2:5] + leftTop_coords

                        pred_final = torch.cat(
                            [
                                coords[:, 1:2] + 1,
                                coords[:, 4:5],
                                coords[:, 3:4],
                                coords[:, 2:3],
                                h_val,
                            ],
                            dim=1,
                        )
                        # print("no haha")
                        # print(pred_final)
                        return pred_final
                    except:
                        # print('haha')
                        return None
                        return torch.zeros([0, 5]).cuda()

            # load trained checkpoints to model
            model = UNetTest.load_from_checkpoint(args.checkpoints)

            # model = UNetTest().model
            model.eval()
            # Support gpu_id formats like 1, "1", "0,1", "[1]", or [1].
            _raw_gpu = args.gpu_id
            if isinstance(_raw_gpu, (list, tuple)):
                _gpu_ids = [int(g) for g in _raw_gpu]
            else:
                _text_gpu = str(_raw_gpu).strip()
                if _text_gpu.startswith("[") and _text_gpu.endswith("]"):
                    try:
                        _parsed = ast.literal_eval(_text_gpu)
                        if isinstance(_parsed, (list, tuple)):
                            _gpu_ids = [int(g) for g in _parsed]
                        else:
                            _gpu_ids = [int(_parsed)]
                    except Exception:
                        _gpu_ids = [
                            int(g.strip())
                            for g in _text_gpu[1:-1].split(",")
                            if g.strip()
                        ]
                elif "," in _text_gpu:
                    _gpu_ids = [
                        int(g.strip()) for g in _text_gpu.split(",") if g.strip()
                    ]
                else:
                    _gpu_ids = [int(_text_gpu)]

            _strategy = "ddp" if len(_gpu_ids) > 1 else "auto"
            runner = Trainer(
                accelerator="gpu", devices=_gpu_ids, strategy=_strategy
            )
            # os.makedirs(f"result/{dataset}/{model_name}/", exist_ok=True)

            runner.test(model=model)

    end_time = time.time()
    used_time = end_time - start_time
    # save_path = "/".join(args.checkpoints.split("/")[:-2]) + f"/{args.out_name}"
    # os.makedirs(save_path, exist_ok=True)
    pad_size = args.pad_size[0]

print("*" * 100)
print("Testing Finished!")
print("*" * 100)
# if stdout is not None:
#     sys.stdout = save_stdout
#     sys.stderr = save_stderr


24092001
(500, 1024, 1024)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


****************************************************************************************************
num_name: /data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/coords/num_name.csv
base_path:/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60
coord_path:/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/coords
tomo_path:/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/data_std
label_path:/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/gaussian11
ocp_path:/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/hsp60/data_ocp
label_name:gaussian11
coord_format:.coords
tomo_format:.mrc
norm_type:standardization
ocp_name:data_ocp
****************************************************************************************************
   0         1
0  0  24092018
1  1  24092028
2  2  24092024
3

In [49]:
cfg

{'dset_name': 'ribosome_fold2_train_fold1_test_24',
 'base_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome',
 'coord_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/coords',
 'coord_format': '.coords',
 'tomo_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/data_std',
 'tomo_format': '.mrc',
 'num_cls': 1,
 'label_type': 'gaussian',
 'label_diameter': 25,
 'ocp_type': 'sphere',
 'ocp_diameter': '41',
 'norm_type': 'standardization',
 'label_name': 'gaussian25',
 'label_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/gaussian25',
 'ocp_name': 'data_ocp',
 'ocp_path': '/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/data_ocp',
 'model_name': 'ResUNet',
 'train_set_ids': '0,2,4,6,8,10,12',
 'val_set_ids': '1,3,5,7,9,11,13',
 'batch_size': 8,
 'patch_size': 